## NB-04 — Vector Database and Indexing

With the knowledge base chunked and tagged, this notebook builds the FAISS
vector index that powers all retrieval in CatalogueIQ. You will embed every
chunk from NB-03, store the vectors in FAISS, run five test queries (one per
query type), inspect the raw retrieved chunks, and implement an embedding cache
so the full catalogue does not need to be re-embedded on every restart.
Inspecting retrieval quality *before* connecting generation is one of the most
important engineering habits in RAG development.

### Concepts Covered

- Generating embeddings for all ShopSmart India chunks with SentenceTransformer
- Building a FAISS flat L2 index and understanding cosine similarity search
- Running five test queries (one per CatalogueIQ query type) and reading raw retrieved chunks
- Diagnosing retrieval failures: why a comparative query may return only one product
- Embedding cache: saving the FAISS index to `data/index/faiss.index` and timing load vs. rebuild
- Why retrieval quality at this stage determines the ceiling for generation quality in NB-05

In [ ]:
# ── SETUP ──────────────────────────────────────────────────────
import os
import json
import time
import pickle
from pathlib import Path
import numpy as np
import faiss
from dotenv import load_dotenv
from langchain.schema import Document
from sentence_transformers import SentenceTransformer

load_dotenv()

MODEL = "claude-sonnet-4-6"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
INDEX_DIR = Path("data/index")
os.makedirs(INDEX_DIR, exist_ok=True)
os.makedirs("output", exist_ok=True)

print(f"FAISS version: {faiss.__version__}")
print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"Index directory: {INDEX_DIR.resolve()}")


### Step 1 — Load Chunks from NB-03

We load the serialised chunks from `output/all_chunks.json`. If NB-03 has not
been run yet, a small set of demo chunks is created so this notebook can still
execute.

In [ ]:
# ── LOAD CHUNKS ────────────────────────────────────────────────

chunks_path = Path("output/all_chunks.json")
if chunks_path.exists():
    with open(chunks_path, encoding="utf-8") as f:
        raw = json.load(f)
    all_docs = [Document(page_content=r["page_content"], metadata=r["metadata"]) for r in raw]
    print(f"Loaded {len(all_docs)} chunks from {chunks_path}")
else:
    print("⚠️  output/all_chunks.json not found. Run NB-03 first, or using demo chunks.")
    # Minimal demo chunks so NB-04 can still execute standalone
    all_docs = [
        Document(
            page_content="Product: boAt Airdopes 141\nBrand: boAt\nCategory: Electronics > Earbuds\n"
                         "Price: ₹1299.00\nSpecs: Battery: 42h total|ANC: No|Connectivity: Bluetooth 5.1|Water Resistance: IPX4\n"
                         "Warranty: 12 months\nReturnable: Yes (7-day window)\nSeller ID: SELL-042 (Verified: Yes)",
            metadata={"source_file": "products.csv", "file_type": "csv", "product_id": "ELEC-001",
                      "category": "Electronics", "price_inr": 1299.0}
        ),
        Document(
            page_content="Product: Sony WF-C500\nBrand: Sony\nCategory: Electronics > Earbuds\n"
                         "Price: ₹2990.00\nSpecs: Battery: 20h total|ANC: No|Connectivity: Bluetooth 5.0|Water Resistance: IPX4\n"
                         "Warranty: 12 months\nReturnable: Yes (7-day window)\nSeller ID: SELL-011 (Verified: Yes)",
            metadata={"source_file": "products.csv", "file_type": "csv", "product_id": "ELEC-002",
                      "category": "Electronics", "price_inr": 2990.0}
        ),
        Document(
            page_content="Product: Jabra Evolve2 Buds\nBrand: Jabra\nCategory: Electronics > Earbuds\n"
                         "Price: ₹8999.00\nSpecs: Battery: 33h total|ANC: Yes|Connectivity: Bluetooth 5.2|Water Resistance: IP57\n"
                         "Warranty: 24 months\nReturnable: Yes (7-day window)\nSeller ID: SELL-011 (Verified: Yes)",
            metadata={"source_file": "products.csv", "file_type": "csv", "product_id": "ELEC-003",
                      "category": "Electronics", "price_inr": 8999.0}
        ),
        Document(
            page_content="## Return Windows\nStandard Return Window: 7 days from delivery for Electronics.\n"
                         "Extended Return Window: 30 days from delivery for Fashion and Home & Kitchen.",
            metadata={"source_file": "returns_policy.md", "file_type": "markdown",
                      "section_heading": "Return Windows"}
        ),
        Document(
            page_content="## Image Guidelines\nElectronics Category: Minimum resolution 1000x1000 pixels. "
                         "Background: Pure white (RGB 255,255,255). Minimum 4 images, maximum 12. "
                         "Prohibited: watermarks, seller logos, text overlays on hero image.",
            metadata={"source_file": "seller_onboarding.md", "file_type": "markdown",
                      "section_heading": "Image Guidelines"}
        ),
        Document(
            page_content="Q21. How do I initiate a return? Go to My Orders, select the order, "
                         "and click Return or Replace. Choose your reason and preferred resolution.",
            metadata={"source_file": "buyer_faq.html", "file_type": "html",
                      "question_number": "faq-021", "section_heading": "Returns & Refunds"}
        ),
    ]
    print(f"Using {len(all_docs)} demo chunks.")


### Step 2 — Generate Embeddings

We embed all chunks using `SentenceTransformer`. For a 200-row product catalogue
plus policy documents this takes ~10–30 seconds on CPU. We time the operation
to understand baseline embedding cost before introducing the cache.

In [ ]:
# ── EMBED ALL CHUNKS ─────────────────────────────────────────

print(f"Loading embedding model: {EMBEDDING_MODEL}")
embedder = SentenceTransformer(EMBEDDING_MODEL)
embedding_dim = embedder.get_sentence_embedding_dimension()
print(f"Embedding dimension: {embedding_dim}")

# Extract text content from all documents
texts = [doc.page_content for doc in all_docs]

print(f"\nEmbedding {len(texts)} chunks...")
t0 = time.time()
# batch_size=64 is efficient for all-MiniLM-L6-v2 on CPU
embeddings = embedder.encode(texts, batch_size=64, show_progress_bar=True,
                              convert_to_numpy=True)
embed_time = time.time() - t0

print(f"\nEmbedding complete.")
print(f"  Time: {embed_time:.2f}s")
print(f"  Matrix shape: {embeddings.shape}  (chunks × embedding_dim)")
print(f"  Throughput: {len(texts)/embed_time:.1f} chunks/sec")


### Step 3 — Build FAISS Index and Save to Disk

We use a flat L2 index (`IndexFlatL2`) normalised to enable cosine similarity
search. The index is saved to `data/index/faiss.index` alongside a pickle of
the document metadata. The next run loads from disk instead of re-embedding.

In [ ]:
# ── BUILD FAISS INDEX ────────────────────────────────────────

# Normalise embeddings so L2 distance ≡ cosine distance
faiss.normalize_L2(embeddings)

# IndexFlatL2: exact brute-force search (fine for <10k chunks)
index = faiss.IndexFlatL2(embedding_dim)
index.add(embeddings)
print(f"FAISS index built. Total vectors: {index.ntotal}")

# ── Save index to disk ────────────────────────────────────────
index_path = INDEX_DIR / "faiss.index"
meta_path  = INDEX_DIR / "metadata.pkl"

t0 = time.time()
faiss.write_index(index, str(index_path))
# Save document metadata (page_content + metadata dict) as pickle
with open(meta_path, "wb") as f:
    pickle.dump(all_docs, f)
save_time = time.time() - t0

print(f"\nIndex saved to {index_path}")
print(f"Metadata saved to {meta_path}")
print(f"Save time: {save_time*1000:.1f} ms")

# ── Reload from disk and time it ─────────────────────────────
t0 = time.time()
index_loaded = faiss.read_index(str(index_path))
with open(meta_path, "rb") as f:
    docs_loaded = pickle.load(f)
load_time = time.time() - t0

print(f"\nIndex reloaded from disk.")
print(f"Load time: {load_time*1000:.1f} ms  (vs {embed_time*1000:.0f} ms to re-embed)")
print(f"Speedup: {embed_time / max(load_time, 0.001):.0f}x faster to load from cache")


### Step 4 — Five Test Queries (One per CatalogueIQ Query Type)

Before connecting generation, we inspect raw retrieval results for all five
query types. This is the most important debugging step: if the wrong chunks
come back here, generation cannot compensate.

In [ ]:
# ── RETRIEVAL FUNCTION ────────────────────────────────────────

def retrieve(query: str, top_k: int = 3) -> list[tuple[Document, float]]:
    """
    Embed query, search FAISS, return (Document, cosine_similarity) tuples.
    Uses the cached index loaded from disk.
    """
    q_vec = embedder.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_vec)                       # normalise for cosine sim
    distances, indices = index_loaded.search(q_vec, top_k)
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if idx == -1:                               # FAISS returns -1 for empty slots
            continue
        cosine_sim = 1 - dist / 2                  # convert L2 to cosine sim
        results.append((docs_loaded[idx], cosine_sim))
    return results

# ── Five test queries ──────────────────────────────────────────

test_queries = [
    ("product_factual",          "battery life of boAt Airdopes earbuds"),
    ("policy_eligibility",       "can I return a fashion item bought on sale"),
    ("comparative_recommendation","wireless earbuds under 3000 rupees with noise cancellation"),
    ("seller_policy",            "image requirements for electronics listings"),
    ("multi_hop",                "smartwatch stopped working after 45 days third party seller"),
]

print("─" * 70)
for query_type, query in test_queries:
    print(f"\n── Query Type: {query_type} ──────────────────────────────────")
    print(f"   Query: "{query}"")
    results = retrieve(query, top_k=3)
    for rank, (doc, sim) in enumerate(results, 1):
        source = doc.metadata.get("source_file", "unknown")
        pid    = doc.metadata.get("product_id", "")
        sec    = doc.metadata.get("section_heading", "")
        ref    = f"product_id={pid}" if pid else f"section={sec}"
        print(f"   Rank {rank} | sim={sim:.4f} | {source} | {ref}")
        print(f"           {doc.page_content[:120].replace(chr(10), ' ')}…")
print("\n─" * 70)
print("\n✅ Inspect the results above:")
print("   - Comparative query: did multiple products appear in the top 3?")
print("   - Multi-hop query: did chunks from multiple documents appear?")
print("   - If only 1 product for comparative → increase top_k or fix chunking.")


In [ ]:
# ── EXPERIMENT ────────────────────────────────────────────────

# EXERCISE 1 — Vary top_k for the comparative recommendation query
# Re-run retrieve("wireless earbuds under 3000 rupees with noise cancellation", top_k=5)
# and top_k=10. At which k do you first see 3 distinct earbuds products?
# What is the cosine similarity score of the 5th result? Is it still relevant?

# EXERCISE 2 — Diagnose a retrieval failure
# Run retrieve("sasta earphone") — a Hinglish query.
# Does boAt Airdopes 141 appear in the results?
# Now run retrieve("cheap earphones India").
# Compare cosine similarity scores. How much does query language affect recall?

# EXERCISE 3 — Category metadata filter
# Implement a filtered retrieval that only returns chunks where
# metadata["category"] == "Electronics".
# Compare the top-3 results for "best product under 2000" with and without the filter.
# Does filtering improve precision? Does it risk missing cross-category answers?

print("Exercises ready. Call retrieve() with different queries and top_k values.")


### Key Takeaway

The FAISS index is the retrieval engine of CatalogueIQ. Loading from disk is
orders of magnitude faster than re-embedding — a critical performance win for
a 200-product catalogue. More importantly, inspecting raw retrieval results
*before* connecting generation reveals problems that generation cannot fix:
if a comparative query returns only one product at this stage, no amount of
prompt engineering will produce a multi-product comparison.

In **NB-05** you will connect the FAISS index to Claude using LangChain's
`RetrievalQA` chain, format retrieved chunks as citations, and write the system
prompt that keeps CatalogueIQ grounded in its knowledge base.